# **Notebook 4: RAG Implementation**
## Capstone: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
from pathlib import Path
import json, pickle
BASE_DIR=Path.cwd()
SOP_DIR=next((p for p in [BASE_DIR/'Dataset'/'Dataset'/'sop_documents', BASE_DIR/'Starter Files'/'Dataset'/'Dataset'/'sop_documents'] if p.exists()), BASE_DIR/'Dataset'/'Dataset'/'sop_documents')
try:
    from langchain_community.document_loaders import TextLoader
    docs=[]
    for path in sorted(SOP_DIR.glob('*.md')):
        loaded=TextLoader(str(path), encoding='utf-8').load()
        for doc in loaded: doc.metadata['intent']=Path(doc.metadata.get('source', path)).stem
        docs.extend(loaded)
except Exception as exc:
    print('Using lightweight local Document objects:', repr(exc))
    from dataclasses import dataclass
    @dataclass
    class Document:
        page_content: str
        metadata: dict
    docs=[Document(p.read_text(encoding='utf-8', errors='replace'), {'source':str(p), 'intent':p.stem}) for p in sorted(SOP_DIR.glob('*.md'))]
try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    sample_embedding=embeddings.embed_query('refund policy'); embedding_backend='HuggingFaceEmbeddings'
except Exception as exc:
    print('Using TF-IDF embeddings fallback:', repr(exc))
    from sklearn.feature_extraction.text import TfidfVectorizer
    embeddings=TfidfVectorizer(stop_words='english')
    doc_matrix=embeddings.fit_transform([d.page_content for d in docs])
    sample_embedding=embeddings.transform(['refund policy']).toarray()[0].tolist(); embedding_backend='TfidfVectorizer'
print(f'Loaded {len(docs)} SOP docs')
print('Embedding backend:', embedding_backend)
print('Sample embedding length:', len(sample_embedding))


Using lightweight local Document objects: ModuleNotFoundError("No module named 'langchain_community'")
Using TF-IDF embeddings fallback: ModuleNotFoundError("No module named 'langchain_community'")


Loaded 13 SOP docs
Embedding backend: TfidfVectorizer
Sample embedding length: 797


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [2]:
CHROMA_DIR=Path('chroma_db'); CHROMA_DIR.mkdir(exist_ok=True)
try:
    from langchain_community.vectorstores import Chroma
    vector_db=Chroma.from_documents(docs, embeddings, persist_directory=str(CHROMA_DIR))
    collection_count=vector_db._collection.count()
except Exception as exc:
    print('Using local TF-IDF vector store fallback:', repr(exc))
    from sklearn.metrics.pairwise import cosine_similarity
    class LocalVectorDB:
        def __init__(self, docs, vectorizer, matrix): self.docs=docs; self.vectorizer=vectorizer; self.matrix=matrix
        def similarity_search(self, query, k=1):
            scores=cosine_similarity(self.vectorizer.transform([query]), self.matrix).ravel(); top=scores.argsort()[::-1][:k]
            return [self.docs[i] for i in top]
        class _Collection:
            def __init__(self,n): self.n=n
            def count(self): return self.n
        @property
        def _collection(self): return self._Collection(len(self.docs))
    vector_db=LocalVectorDB(docs, embeddings, doc_matrix); collection_count=vector_db._collection.count()
    with open(CHROMA_DIR/'fallback_vector_db.pkl','wb') as f: pickle.dump({'texts':[d.page_content for d in docs], 'metadata':[d.metadata for d in docs]}, f)
print('Vector index document count:', collection_count)
print('Top refund retrieval:', vector_db.similarity_search('refund', k=1)[0].metadata)


Using local TF-IDF vector store fallback: ModuleNotFoundError("No module named 'langchain_community'")
Vector index document count: 13
Top refund retrieval: {'source': 'C:\\Users\\sysadmin\\Downloads\\Capestone project\\Starter Files\\Dataset\\Dataset\\sop_documents\\refund_policy.md', 'intent': 'refund_policy'}


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [3]:
outputs_path=Path('outputs.json')
outputs=json.loads(outputs_path.read_text(encoding='utf-8')) if outputs_path.exists() else {}
test_query=outputs.get('test_query','My package still has not arrived and the tracking page has not moved. When will I get it?')
retrieved_doc=vector_db.similarity_search(test_query, k=1)[0]
context=retrieved_doc.page_content
try: model_available
except NameError: model_available=False
def generate_with_context(query, context):
    if model_available:
        messages=[{'role':'system','content':f'Answer strictly using this SOP:\n{context}'},{'role':'user','content':query}]
        prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs=tokenizer(prompt, return_tensors='pt').to(model.device)
        outputs=model.generate(**inputs, max_new_tokens=180, do_sample=False, temperature=None, top_p=None)
        return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    title=context.splitlines()[0].replace('#','').strip()
    body=' '.join(line.strip() for line in context.splitlines() if line.strip() and not line.startswith('#'))
    return f'According to the {title} SOP: {body[:700]}'
naive_rag_output=generate_with_context(test_query, context)
print('Retrieved SOP:', retrieved_doc.metadata)
print('Naive RAG output:\n', naive_rag_output)


Retrieved SOP: {'source': 'C:\\Users\\sysadmin\\Downloads\\Capestone project\\Starter Files\\Dataset\\Dataset\\sop_documents\\order_tracking.md', 'intent': 'order_tracking'}
Naive RAG output:
 According to the Order Tracking SOP: Helps customers locate an order and interpret its status. Orders that have clearly exceeded their delivery window follow the shipping delays procedure. - **Processing:** payment confirmed, not yet packed. - **Preparing to ship:** packed, awaiting carrier pickup. - **In transit:** carrier has the parcel; tracking events update as it moves. - **Out for delivery:** on the vehicle for delivery that day. - **Delivered:** carrier marked the parcel delivered. - **Exception:** address issue, failed delivery, or held parcel needing action. Locate the order by its identifier and the registered email. Provide the tracking number and carrier link. Tracking can take up to 24 hours to show the first scan 


---
## Save Artifacts for Downstream Notebooks

In [4]:
outputs.update({'test_query':test_query,'naive_rag_retrieved_source':retrieved_doc.metadata.get('source',''),'naive_rag_retrieved_intent':retrieved_doc.metadata.get('intent', Path(retrieved_doc.metadata.get('source','')).stem),'naive_rag_output':naive_rag_output})
outputs_path.write_text(json.dumps(outputs, indent=2), encoding='utf-8')
print(f'Updated {outputs_path.resolve()} with naive_rag_output')
print(f'Vector index directory exists: {CHROMA_DIR.resolve().exists()} -> {CHROMA_DIR.resolve()}')


Updated C:\Users\sysadmin\Downloads\Capestone project\Starter Files\outputs.json with naive_rag_output
Vector index directory exists: True -> C:\Users\sysadmin\Downloads\Capestone project\Starter Files\chroma_db


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**